<a href="https://colab.research.google.com/github/FinalInvictus/Deep-Mutual-Learning-with-Semi-Aves-Dataset/blob/Sridhar/DeepMutualLearning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
from getpass import getpass

os.environ["KAGGLE_API_TOKEN"] = getpass("Enter your Kaggle API token: ")

Enter your Kaggle API token: ··········


In [6]:
import kagglehub

path = kagglehub.competition_download(
    "semi-inat-2020",
    output_dir="/content/semi_inat_2020",
    force_download=True
)

100%|██████████| 14.3G/14.3G [11:06<00:00, 23.0MB/s]

Extracting files...


In [ ]:
# First peer arch --> Resnet 18
# 2nd peer arch --> Resnet 18
TOTAL_CLASSES = 200
IMG_SIZE = 224
BATCH_SIZE = 32
TEMPERATURE = 1 # VALUE RANGE 1,2,4
KL_WEIGHT = 1 # VALUE RANGE 0,0.5,1,2
EPOCH = 2 # 200 WERE USED IN ORIGINAL PAPER

In [14]:
import torchvision
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
from torchvision import transforms

#creating transforms to make our data suitable
trans=transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
#resized to 224*224 and normalized wrt to Imagenet1k




In [15]:
train_ds=ImageFolder(root="/content/semi_inat_2020/trainval_images/trainval_images",transform=trans)
train_dl=DataLoader(train_ds,batch_size=32,shuffle=True,)
#converting the image folder into data
print(train_ds.classes)

['0', '1', '10', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '11', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '12', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '13', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '14', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '15', '150', '151', '152', '153', '154', '155', '156', '157', '158', '159', '16', '160', '161', '162', '163', '164', '165', '166', '167', '168', '169', '17', '170', '171', '172', '173', '174', '175', '176', '177', '178', '179', '18', '180', '181', '182', '183', '184', '185', '186', '187', '188', '189', '19', '190', '191', '192', '193', '194', '195', '196', '197', '198', '199', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '4', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '5', '50', '51', '52', '53', '54',

In [16]:
imgs,labels=next(iter(train_dl))
print(imgs.shape)
print(labels.shape)
#checking if the data is a valid tensor

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [ ]:
import torch
from torch import nn

class ConvBlock(nn.Module):
  def __init__(self,in_ch,out_ch,stride=1):
    super().__init__()
    self.conv=nn.Conv2d(in_ch,out_ch,3,1,1)
    self.bnorm=nn.BatchNorm2d(out_ch)
    self.conv2=nn.Conv2d(in_ch,out_ch,3,1,1)
    self.bnorm2=nn.BatchNorm2d(out_ch)
    self.relu=nn.ReLU(inplace=True)

    self.shortcut=nn.Identity()
    if stride!=1 or in_ch!=out_ch:
      self.shortcut=nn.Sequential(
          nn.Conv2d(in_ch,out_ch,1,stride,bias=False),
          nn.BatchNorm2d(out_ch),
      )

    def forward(self,x):
      out=self.relu(self.bnorm(self.conv(x)))
      out=self.bnorm2(self.conv2(out))
      out=out+self.shortcut(x)
      return self.relu(out)


In [1]:
from google.colab import drive
drive.mount('/content/SharedData')

MessageError: Error: credential propagation was unsuccessful

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
